<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/BERT_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate seqeval accelerate -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "eriktks/conll2003",
    revision="convert/parquet"
)

print(dataset)

In [ ]:
example = dataset["train"][0]
example

In [ ]:
dataset["train"].features["ner_tags"]

In [ ]:
labels_name = dataset["train"].features["ner_tags"].feature.names
labels_name

In [ ]:
len(labels_name)

In [ ]:
id2label = { i : label for i, label in enumerate(labels_name)}
id2label

In [ ]:
label2id = { label : i for i , label in enumerate(labels_name)}
label2id

In [ ]:
tokens = example["tokens"]
ner_tags = example["ner_tags"]

for token, ner_tag in zip(tokens, ner_tags):
    print(token , "->", labels_name[ner_tag])

In [ ]:
from transformers import AutoTokenizer

model_name="bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
dataset["train"].features

In [ ]:
# take dataset token and map to bert tokenizer

tokenized_input = tokenizer(
    tokens,
    is_split_into_words=True
)

print(tokenized_input)

In [ ]:
bert_tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

print(bert_tokens)

In [ ]:
word_ids = tokenized_input.word_ids()

for token, word_id in zip(bert_tokens, word_ids):
    print(token, "->", word_id)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_input = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    aligned_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_input.word_ids(i)

        previous_word_id = None
        label_ids=[]

        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                label_ids.append(-100)
            else:
                label_ids.append(labels[word_id])

            previous_word_id = word_id

        aligned_labels.append(label_ids)
    tokenized_input["labels"] = aligned_labels
    return tokenized_input


In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
sample = tokenized_dataset["train"][0]

print(sample.keys())
print(len(sample["input_ids"]))
print(len(sample["attention_mask"]))
print(len(sample["labels"]))

In [ ]:
from transformers import AutoModelForTokenClassification


model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(labels_name),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        sentence_predictions = []
        sentence_labels = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                sentence_predictions.append(labels_name[pred_id])
                sentence_labels.append(labels_name[label_id])

        true_predictions.append(sentence_predictions)
        true_labels.append(sentence_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
import wandb

wandb.login()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="bert-ner-conll2003",

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    report_to="wandb",
    run_name="bert-base-cased-conll2003-ner"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
import torch

def predict_ner_words(text, model, tokenizer, id2label):
    model.eval()
    device = next(model.parameters()).device

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        return_offsets_mapping=True
    )

    offset_mapping = encoding.pop("offset_mapping")[0]
    word_ids = encoding.word_ids(batch_index=0)

    model_inputs = {key: value.to(device) for key, value in encoding.items()}

    with torch.no_grad():
        outputs = model(**model_inputs)

    pred_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    word_predictions = []
    seen_word_ids = set()

    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        if word_id in seen_word_ids:
            continue

        seen_word_ids.add(word_id)

        # Bu word_id-yə aid bütün subword-lərin offset-lərini tapırıq
        subword_indices = [
            j for j, wid in enumerate(word_ids)
            if wid == word_id
        ]

        start = int(offset_mapping[subword_indices[0]][0])
        end = int(offset_mapping[subword_indices[-1]][1])

        word = text[start:end]

        label = id2label[int(pred_ids[subword_indices[0]])]

        word_predictions.append((word, label))

    return word_predictions

In [ ]:
 text = "We are learning NLP in Div Academy"
word_preds = predict_ner_words(
    text=text,
    model=trainer.model,
    tokenizer=tokenizer,
    id2label=id2label
)

for word, label in word_preds:
    print(word, "->", label)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np


def plot_ner_confusion_matrix_without_o(trainer, eval_dataset, label_names):
    pred_output = trainer.predict(eval_dataset)

    logits = pred_output.predictions
    true_label_ids = pred_output.label_ids

    pred_label_ids = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_seq, true_seq in zip(pred_label_ids, true_label_ids):
        for pred_id, true_id in zip(pred_seq, true_seq):
            if true_id != -100:
                true_label = label_names[true_id]
                pred_label = label_names[pred_id]

                if true_label != "O":
                    true_labels.append(true_label)
                    pred_labels.append(pred_label)

    entity_labels = [label for label in label_names if label != "O"]

    cm_norm = confusion_matrix(
        true_labels,
        pred_labels,
        labels=entity_labels,
        normalize="true"
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_norm,
        display_labels=entity_labels
    )

    fig, ax = plt.subplots(figsize=(9, 9))
    disp.plot(
        ax=ax,
        xticks_rotation=45,
        values_format=".3f"
    )

    plt.title("Normalized NER Confusion Matrix Without O Label")
    plt.show()

In [ ]:
plot_ner_confusion_matrix_without_o(
    trainer=trainer,
    eval_dataset=tokenized_dataset["validation"],
    label_names=labels_name
)